# Architecture Transformer

Ce notebook reprend le cours sur les réseaux récurrents, les LSTM et les Transformers, et le relie à MaintiDoc.

MaintiDoc est un assistant **documentaire**. Il ne pilote aucun équipement. Les modèles de langue qu'il peut appeler sont des Transformers ; leurs poids ne sont pas téléchargés par défaut. Ici, on n'entraîne qu'un petit exemple sur CPU : prédire la valeur suivante d'une sinusoïde à partir des dix précédentes.

Les images du support d'origine ne sont pas recopiées. Le code exécutable est dans `notebooks/sequence_models.py`.


## 1. RNN

Un réseau récurrent lit une séquence dans l'ordre. À l'instant \(t\) il reçoit l'entrée \(x_t\) et l'état précédent \(h_{t-1}\), puis calcule

\[
h_t = \tanh(W_{ih} x_t + W_{hh} h_{t-1} + b)
\]

Les mêmes poids servent à chaque instant : les cases du schéma sont **une seule couche déroulée**, pas des couches distinctes.

| Élément | Rôle |
| --- | --- |
| `input_size=1` | Une valeur observée à chaque instant |
| longueur 10 | Dix pas de la sinusoïde |
| `hidden_size=16` | Mémoire de 16 nombres |
| `nn.Linear(16, 1)` | Lecture du **dernier** état seulement |

Formes, avec \(B\) séquences dans le lot : \((B, 10, 1) \rightarrow (B, 10, 16) \rightarrow (B, 16) \rightarrow (B, 1)\).

Limite : sur une longue séquence, les gradients peuvent disparaître ou exploser. Le LSTM, puis l'attention, répondent à cette limite. L'état caché part de zéros quand on n'en fournit pas.


In [ ]:
import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
if not (ROOT / "notebooks").is_dir() and (ROOT.parent / "notebooks").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

torch.manual_seed(42)
from notebooks.sequence_models import SimpleRNN, make_windows, train_and_eval
import inspect

print(inspect.getsource(SimpleRNN))
periode, serie, X_train, y_train, X_test, y_test = make_windows()
print("formes", tuple(X_train.shape), tuple(y_train.shape), tuple(X_test.shape))
resultat_rnn = train_and_eval(SimpleRNN(), X_train, y_train, X_test, y_test, epochs=300)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

valeurs_reelles = resultat_rnn["y_test"].detach().cpu().numpy().reshape(-1)
valeurs_predites = resultat_rnn["predictions"].detach().cpu().numpy().reshape(-1)
plt.figure(figsize=(12, 5))
plt.plot(valeurs_reelles, label="Valeurs réelles", color="blue", linewidth=2)
plt.plot(valeurs_predites, label="Prédictions du RNN", color="red", linestyle="--", linewidth=2)
plt.title("RNN : valeurs réelles et prédictions")
plt.xlabel("Indice de l'échantillon de test")
plt.ylabel("Valeur")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2. LSTM

Le LSTM garde un état de cellule \(c_t\) en plus de l'état caché \(h_t\). Trois portes apprises décident quoi oublier, quoi écrire et quoi exposer. Les poids sont plus nombreux que ceux du RNN, et les dépendances un peu plus longues passent mieux, sans garantie de tout mémoriser.

Sur le même jeu, `nn.LSTM(input_size=1, hidden_size=16, num_layers=1, batch_first=True)` lit les dix pas. Sans état initial, \(h_0\) et \(c_0\) valent zéro. La couche linéaire lit encore uniquement le dernier état caché.

À retenir : le LSTM apprend quelles informations conserver, ajouter et exposer.


In [ ]:
import inspect
from notebooks.sequence_models import SimpleLSTM

torch.manual_seed(42)
print(inspect.getsource(SimpleLSTM))
resultat_lstm = train_and_eval(SimpleLSTM(), X_train, y_train, X_test, y_test, epochs=300)


In [ ]:
valeurs_reelles = resultat_lstm['y_test'].detach().cpu().numpy().reshape(-1)
valeurs_predites = resultat_lstm['predictions'].detach().cpu().numpy().reshape(-1)
plt.figure(figsize=(12, 5))
plt.plot(valeurs_reelles, label='Valeurs réelles', color='blue', linewidth=2)
plt.plot(valeurs_predites, label='Prédictions du LSTM', color='red', linestyle='--', linewidth=2)
plt.title('LSTM : valeurs réelles et prédictions')
plt.xlabel("Indice de l'échantillon de test")
plt.ylabel('Valeur')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Encodeur Transformer

Le Transformer ne fait pas passer une mémoire d'un instant au suivant. Chaque position est mise en relation avec les autres par l'attention, et les positions peuvent être calculées en parallèle.

Le mini-modèle du cours, sur la même sinusoïde :

1. une couche linéaire projette la valeur scalaire dans un vecteur de taille 16 ;
2. un vecteur de position appris est ajouté (les positions ne sont pas dans l'ordre du RNN) ;
3. un bloc `TransformerEncoderLayer` (deux têtes, réseau interne de taille 32, dropout nul) calcule l'auto-attention ;
4. la dernière position est lue par une couche linéaire.

Il n'y a pas de décodeur. Le coût de l'attention croît avec le carré de la longueur, ici 10, donc négligeable. C'est ce mécanisme d'attention que les modèles de langue de MaintiDoc utilisent. Le prototype ne télécharge pas ces poids : le générateur par défaut reste déterministe.


In [ ]:
import inspect
from notebooks.sequence_models import MiniTransformer

torch.manual_seed(42)
print(inspect.getsource(MiniTransformer))
resultat_transformer = train_and_eval(
    MiniTransformer(longueur_max=10), X_train, y_train, X_test, y_test, epochs=300
)


In [ ]:
valeurs_reelles = resultat_transformer['y_test'].detach().cpu().numpy().reshape(-1)
valeurs_predites = resultat_transformer['predictions'].detach().cpu().numpy().reshape(-1)
plt.figure(figsize=(12, 5))
plt.plot(valeurs_reelles, label='Valeurs réelles', color='blue', linewidth=2)
plt.plot(valeurs_predites, label='Prédictions du Transformer', color='red', linestyle='--', linewidth=2)
plt.title('Transformer : valeurs réelles et prédictions')
plt.xlabel("Indice de l'échantillon de test")
plt.ylabel('Valeur')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Décodeur causal, à la place de GPT-2

Le cours charge `gpt2` avec `AutoModelForCausalLM` et génère la suite de « Artificial intelligence can help people ». GPT-2 est un décodeur : l'attention est **causale** (une position ne voit pas le futur), la génération est autorégressive, et `generate` ajoute des tokens un par un (`max_new_tokens`, `temperature`, `top_p`).

Ce dépôt ne télécharge pas GPT-2. La cellule entraîne un décodeur d'une couche sur **cette seule phrase**, au niveau des caractères, puis prolonge l'amorce. Le texte obtenu montre le mécanisme. Il ne reprend pas les connaissances de GPT-2.


In [ ]:
from notebooks.sequence_models import prolonger_phrase

texte_genere = prolonger_phrase(epochs=300)
print('Texte généré :')
print(texte_genere)


## 5. Masque bidirectionnel, à la place de BERT

Le cours charge `bert-base-multilingual-cased` et complète « Paris est la [MASK] de la France ». BERT est un encodeur : l'attention voit le contexte **à gauche et à droite** du masque. GPT, lui, ne voit que le passé.

Ce dépôt ne télécharge pas BERT. La cellule entraîne un encodeur d'une couche sur trois phrases du même moule pour retrouver le mot « capitale ». Une probabilité élevée signifie que le jouet a mémorisé ces phrases, pas que le modèle connaît la géographie.


In [ ]:
from notebooks.sequence_models import proposer_masque

texte = 'Paris est la [MASK] de la France'
print('Texte :', texte)
print('Propositions du modèle local :')
for token, probabilite in proposer_masque(epochs=400):
    print(f'{probabilite:.2%} | {texte.replace("[MASK]", token)}')
